# N1 · Forward PINN: the cantilever
**Physics-informed machine learning for civil engineers · hands-on 1 · about 35 minutes**

A network that has **never seen a measurement** predicts the deflection, moment and shear of a cantilever, from the governing equation alone.

**The problem (scaled units).** `EI w'''' = q` on 0 < x < L, fixed at x = 0 and free at x = L. In scaled units this is simply

`w''''/8 = 1` on (0, 1),  `w(0) = 0`, `w'(0) = 0` (fixed end),  `w''(1) = 0`, `w'''(1) = 0` (free end: no moment, no shear).

(The deflection is measured in units of the tip deflection `q L⁴/8EI`, so a correct beam runs from 0 to 1. That puts the same 8 into the equation, and `w''''/8` is simply **the load the network implies**, in units of the reference load.)

**The idea.** We build a network `w(x)` and a loss that is zero only when the network satisfies the equation and the four boundary conditions. Then we train.

| Part | What you do | Minutes |
|---|---|---|
| A | scaling; the network and derivative helper (from N0) | 3 |
| B | **TODO 1, 2, 3**: derivatives, physics loss, boundary loss; train; plot | 20 |
| C | **TODO 4**: check the PINN against statics like an engineer | 5 |
| class experiment 1 | your birth month sets the balance of the loss; two numbers go into the class form | 3 |
| D | *extra:* why we scale; how much does the loss weight matter? | 5 |
| E | *extra:* back to three sensors: data-only network against PINN | 5 |

> **Student version.** Fill in the cells marked `TODO` (look for the word `None`). Each TODO has (1) a short explanation above it, (2) a **hint** you can open, and (3) a **check cell** below it that prints `PASSED` when your code works, or `NOT YET` with a hint. Run the cells in order: the training cell will stop with a clear message if a TODO is not finished. The full solution is in `N1_solution.ipynb`: look at it *after* you have tried.

## Set-up
Libraries, the beam and its exact solution (same cells as N0; see N0 for the line-by-line explanation).

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn                    # same libraries as in N0

torch.set_default_dtype(torch.float64)   # 64-bit numbers: needed for clean 4th derivatives (see N0 part 3)
print("PyTorch", torch.__version__, "| numbers are", torch.get_default_dtype())

PyTorch 2.5.1+cu121 | numbers are torch.float64


In [ ]:
# ---- The beam, in SI units -------------------------------------------------------------------------
EI   = 1.755e7        # bending stiffness of an IPE 300, N m^2
L    = 3.0            # length of the cantilever, m
Q_SI = 10e3           # uniform load, N/m  (= 10 kN/m)
P_SI = 30e3           # point load, N (= 30 kN = q0 L): P = 1 in scaled units means exactly this

# ---- Scaling: we work with numbers of order 1 (N1 part D shows why this matters) -------------------
# x is measured in units of L, and the deflection in units of W0 = the tip deflection of THIS beam
# under the reference load, so that a correct deflection runs from 0 at the clamp to 1.0 at the tip.
W0 = Q_SI * L**4 / (8 * EI)   # unit of deflection = 5.8 mm  (the textbook tip deflection q L^4 / 8EI)
M0 = EI * W0 / L**2           # what one unit of w''  is worth = 11.25 kN m
F0 = EI * W0 / L**3           # what one unit of w''' is worth = 3.75 kN

def to_mm(w):    return w * W0 * 1e3     # scaled deflection -> millimetres
def to_kNm(M):   return M * M0 / 1e3     # scaled moment     -> kN m
def to_kN(V):    return V * F0 / 1e3     # scaled force      -> kN

# ---- Exact (textbook) solution for the uniform load, in scaled units: q = 1 ----------------------------
# Fixed end at x = 0, free end at x = 1.
def w_unif(x): return x**2 * (6 - 4*x + x**2) / 3      # deflection  w:  0 at the clamp, 1.0 at the tip
def M_unif(x): return 4 * (1 - x)**2                    # bending moment  M = w''   (to_kNm -> kN m)
def V_unif(x): return 8 * (1 - x)                       # shear force     V = -w'''  (to_kN  -> kN)

# The beam equation in these units is  w'''' = 8 q.  The 8 is the same 8 as in the tip deflection q L^4 / 8EI.
# We always divide by it and use the form  w''''/8 = q, which reads: the load the network implies = the load we assume.
GRID = np.linspace(0, 1, 201)                           # 201 points along the beam, used for plots

# ---- Two numbers to judge a fit ------------------------------------------------------------------------
def r2(y, yhat):      return 1 - np.sum((y - yhat)**2) / np.sum((y - y.mean())**2)   # R^2: 1.0 = perfect fit
def rel_l2(y, yhat):  return np.linalg.norm(y - yhat) / np.linalg.norm(y)             # relative error: 0.1 = 10 % off

# ---- Virtual sensor readings: exact deflection + random noise -----------------------------------------------
def make_sensors(n, x_first=0.05, x_last=1.0, noise=0.10, seed=0, q=1.0, P=0.0):
    """n virtual sensors between x_first and x_last. noise = standard deviation as a fraction of the tip deflection."""
    xs = np.linspace(x_first, x_last, n)
    w_exact = q * w_unif(xs)                  # exact deflection at the sensors
    w_tip = q * w_unif(1.0)                   # exact tip deflection (the noise is a fraction of this)
    if P != 0:                                # a point load as well (from N2 part D on)
        w_exact = w_exact + P * w_pt(xs)
        w_tip = w_tip + P * w_pt(1.0)
    ys = w_exact + noise * w_tip * np.random.RandomState(seed).randn(n)    # same seed = same noise every run
    return xs, ys

**Plumbing.** Run the next cell once (its code is hidden: click the small `…` to look at it). It only sets up plot style, saving figures for the slides, and the **check functions** that test your TODOs. You do not need to read it.

In [ ]:
import os, json
# colours and plot style (cosmetic)
NAVY, ORANGE, GREY, TEAL, INK = "#002147", "#E9560D", "#9AA5B1", "#1B8C84", "#333F48"
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#5F6B74",
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": NAVY, "legend.frameon": False,
    "figure.dpi": 100, "axes.grid": True, "grid.color": "#E3E8EE", "grid.linewidth": 0.8})

# show(fig, name): tidy the layout, display the figure, and (only when run_all.py runs the notebook) save it as a slide figure
FIG_DIR = os.environ.get("PIML_FIG_DIR")
def show(fig, name=None):
    fig.tight_layout()
    if FIG_DIR and name:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

# record(key, value) keeps a number for the slides; save_results writes them to results_N?.json (used by run_all.py)
RESULTS = {}
def record(key, value):
    RESULTS[key] = value if isinstance(value, (list, str)) else float(value)
    return value
def save_results(name):
    with open(f"results_{name}.json", "w") as f: json.dump(RESULTS, f, indent=1)

# ---- checks for the TODO cells ---------------------------------------------------------------------------------
STATUS = {}                                    # remembers which TODOs passed
def run_check(label, test, hint="", todo=None):
    """run test() and print PASSED or NOT YET (with a hint). todo = number of the TODO this check belongs to."""
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        if "NoneType" in str(e): hint = "a TODO line still says `None`: replace it with your expression (see the explanation above the cell)"
        else: hint = f"{hint}  [{type(e).__name__}: {e}]".strip()
    print(("PASSED   " if ok else "NOT YET  ") + label + ("" if ok else "\n          hint: " + hint))
    if todo is not None: STATUS[todo] = ok

def need_todos(*numbers):
    """stop with a clear message if a TODO is not finished (called before the training cells)"""
    missing = [n for n in numbers if not STATUS.get(n)]
    if missing:
        raise RuntimeError(f"TODO {missing} is not finished yet. Go back to it: its check cell must print PASSED.")
def check_derivs():
    """TODO 1: w = x^4/24 has w'' = x^2/2 and w'''' = 1"""
    def test_fn(x): return x**4 / 24
    def test():
        xc = torch.tensor([[0.3], [0.7]], requires_grad=True)
        w, w1, w2, w3, w4 = derivs(test_fn, xc)
        w4_ok = np.abs(w4.detach().numpy() - 1).max() < 1e-10
        w2_ok = np.abs(w2.detach().numpy() - xc.detach().numpy() ** 2 / 2).max() < 1e-10
        return w4_ok and w2_ok
    run_check("TODO 1: x^4/24 has w'''' = 1 and w'' = x^2/2", test, "w4 should be 1 everywhere", todo=1)

def exact_deflection(x): return x**2 * (6 - 4*x + x**2) / 3       # the exact solution, as a PyTorch formula

def check_physics_loss():
    """TODO 2: exact solution -> zero physics loss; w = x^5 -> large loss"""
    def wrong_fn(x): return x**5
    xc = torch.rand(50, 1).requires_grad_(True)                    # 50 random test points in (0, 1)
    run_check("TODO 2: the exact solution has zero physics loss", lambda: physics_loss(exact_deflection, xc).item() < 1e-18, todo=2)
    run_check("TODO 2: w = x^5 has a large residual", lambda: physics_loss(wrong_fn, xc).item() > 0.5, todo=2)

def check_bc_loss():
    """TODO 3: exact solution -> zero boundary loss; w = x + x^5 violates w'(0) = 0"""
    def wrong_fn(x): return x + x**5
    run_check("TODO 3: the exact solution has zero boundary loss", lambda: bc_loss(exact_deflection).item() < 1e-18, todo=3)
    run_check("TODO 3: w = x + x^5 violates w'(0) = 0", lambda: bc_loss(wrong_fn).item() > 0.5, todo=3)

def fmt(value, digits=2):
    """number as text; says so if a TODO is not done yet (value is None)"""
    return "(TODO 4 not done yet)" if value is None else f"{value:.{digits}f}"

def check_tip_and_moment():
    """TODO 4: the two numbers read off the PINN agree with statics within 5 %"""
    run_check("TODO 4: tip deflection within 5 % of qL^4/8EI", lambda: abs(tip_mm - exact_tip) / exact_tip < 0.05, todo=4)
    run_check("TODO 4: fixed-end moment within 5 % of qL^2/2", lambda: abs(M0_kNm - exact_M0) / exact_M0 < 0.05, todo=4)

## A · Scaling and the building blocks
With L = 3 m, q = 10 kN/m and EI = 1.755 × 10⁷ N·m², the unit of deflection is the tip deflection `q L⁴ / 8EI` = 5.77 mm, so a correct deflection runs from 0 at the clamp to **1.0** at the tip. In these units the beam equation is `w'''' = 8 q`; we divide by the 8 and read it as `w''''/8 = q`, the load the network implies against the load we assume. Every number the optimiser sees is then of order one. Moment and shear come back in kN·m and kN through `to_kNm` and `to_kN`.

In [ ]:
print(f"unit of deflection  W0 = q L^4 / 8EI = {W0*1e3:.2f} mm,  so w = 1 IS the tip deflection of this beam")
print(f"exact tip deflection  q L^4 / 8 EI = {to_mm(w_unif(1.0)):.2f} mm  = {w_unif(1.0):.2f} in scaled units")
print(f"exact fixed-end moment q L^2 / 2   = {to_kNm(M_unif(0.0)):.1f} kN m")

**Two helpers from N0** (read N0 part A if they look unfamiliar): `make_net` builds the network `1 → 30 → 30 → 1` with tanh, and `d(y, x)` differentiates `y` with respect to `x` using autograd. `create_graph=True` lets us differentiate the result again, which we need four times.

In [5]:
def make_net(width=30, act=nn.Tanh):
    """1 -> 30 -> 30 -> 1, tanh. 1,021 trainable parameters."""
    return nn.Sequential(nn.Linear(1, width), act(), nn.Linear(width, width), act(), nn.Linear(width, 1))

def d(y, x):
    """dy/dx by automatic differentiation, at every point. create_graph=True so the result can be differentiated again."""
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

print("parameters:", sum(p.numel() for p in make_net().parameters()))

parameters: 1021


## B · The PINN
A PINN is built from five pieces. Three of them are your TODOs:

1. a network `w(x)` (done: `make_net`);
2. its derivatives `w'`, `w''`, `w'''`, `w''''` by autodiff → **TODO 1**;
3. the **physics loss**: how badly `w''''/8 = 1` is violated at random *collocation points* inside the beam → **TODO 2**;
4. the **boundary loss**: how badly the four boundary conditions are violated at the two ends → **TODO 3**;
5. the optimiser loop that makes the total loss small (given).

### TODO 1 · Derivatives of the network (four lines)
**What to do.** In `derivs`, build the slope `w1`, moment `w2`, shear `w3` and load `w4` of the network output `w`.

**Why.** The beam equation is `w''''/8 = 1`. To test it we need the fourth derivative of the *network* with respect to `x`. The other derivatives are needed for the boundary conditions (slope at the fixed end, moment and shear at the free end).

**How.** One call of `d` per line, each time on the line above (as in N0 part A2: `w1 = d(w, x)`, then `w2` from `w1`, ...).

<details><summary>Hint (open if stuck)</summary>

`w1 = d(w, x)`. The next three lines follow the same pattern: `w2` is the derivative of `w1`, `w3` of `w2`, `w4` of `w3`.
</details>

In [ ]:
def derivs(model, x):
    """network output w and its first four derivatives with respect to x"""
    w = model(x)                  # deflection (given)
    w1 = None                     # TODO 1a: slope      w'    = d(w, x)
    w2 = None                     # TODO 1b: moment     w''   = derivative of w1
    w3 = None                     # TODO 1c: shear      w'''  = derivative of w2
    w4 = None                     # TODO 1d: load       w'''' = derivative of w3
    return w, w1, w2, w3, w4

**Check 1.** The check feeds `derivs` a function whose derivatives we know by hand: `w = x⁴/24` has `w'' = x²/2` and `w'''' = 1`. It must print `PASSED`.

In [ ]:
check_derivs()

### TODO 2 · The physics loss (two lines)
**What to do.** In `physics_loss`, compute the *residual* of the beam equation at every collocation point and return the mean of its square.

**Why.** If the network satisfies `w''''/8 = load` exactly, the residual `w''''/8 − load` is zero at every point. The mean squared residual is one number that is zero for a perfect solution and grows when the equation is violated. This is the term that brings the *physics* into the loss.

**How.** `residual = w4 / 8 − load`, then square it and take the mean over the points. (`**2` squares a tensor, `.mean()` averages it.)

<details><summary>Hint (open if stuck)</summary>

`residual = w4 / 8 - load` and `return (residual ** 2).mean()`
</details>

In [ ]:
def physics_loss(model, x, load=1.0):
    """mean squared residual of the beam equation  w''''/8 = load  at the collocation points x"""
    w, w1, w2, w3, w4 = derivs(model, x)
    residual = None               # TODO 2a: the load the network implies (w''''/8) minus the load
    return None                   # TODO 2b: the MEAN of the SQUARED residual

**Check 2.** The exact deflection must give (almost) zero physics loss; a wrong function (`w = x⁵`) must give a large one.

In [ ]:
check_physics_loss()

### TODO 3 · The boundary loss (four lines)
**What to do.** The network must respect the supports. Write the four boundary conditions as squared terms; `bc_loss` returns their sum.

**Why.** The physics loss only looks *inside* the beam. Without the supports, many functions satisfy `w''''/8 = 1`. The boundary conditions pick the one that belongs to *this* cantilever.

**How.** The function evaluates the network at two points: row 0 is the **fixed end** (x = 0), row 1 is the **free end** (x = 1). Each quantity (`w`, `w1`, `w2`, `w3`) has shape `(2, 1)`, so `w[0, 0]` means "row 0, column 0" = the deflection at the fixed end.

| condition | meaning | what to square |
|---|---|---|
| `w(0) = 0` | no deflection at the fixed end | `w[0, 0] / 8` |
| `w'(0) = 0` | no rotation at the fixed end | `w1[0, 0] / 8` |
| `w''(1) = 0` | no moment at the free end | `w2[1, 0] / 8` |
| `w'''(1) = 0` | no shear at the free end | `w3[1, 0] / 8` |

Each one is divided by 8 before squaring — the same 8 as in `w''''/8`. These are all derivatives of `w`, so they carry the factor the scaling put there; dividing it out makes each term a fractional error, like the other two losses.

<details><summary>Hint (open if stuck)</summary>

`fixed_deflection = (w[0, 0] / 8) ** 2`. The other three lines are the same with `w1[0, 0]`, `w2[1, 0]` and `w3[1, 0]`.
</details>

In [ ]:
def bc_loss(model):
    """how badly the four boundary conditions are violated (each squared, then summed)"""
    xb = torch.tensor([[0.0], [1.0]], requires_grad=True)      # two boundary points: row 0 = fixed end, row 1 = free end
    w, w1, w2, w3, _ = derivs(model, xb)                       # each of shape (2, 1): index them as [row, 0]
    fixed_deflection = None       # TODO 3a: w(0) must be 0     -> square w[0, 0] / 8
    fixed_slope      = None       # TODO 3b: w'(0) must be 0    -> square w1[0, 0] / 8
    free_moment      = None       # TODO 3c: w''(1) must be 0   -> square w2[1, 0] / 8
    free_shear       = None       # TODO 3d: w'''(1) must be 0  -> square w3[1, 0] / 8
    return fixed_deflection + fixed_slope + free_moment + free_shear

**Check 3.** The exact solution satisfies all four conditions (loss ≈ 0). The function `w = x + x⁵` has slope 1 at the fixed end, so it must be penalised.

In [ ]:
check_bc_loss()

### The training loop (given)
Read the comments: this is the loop from N0 part 1.4, now with a physics loss.

- **100 new random collocation points at every iteration.** The network cannot memorise points, it has to satisfy the equation everywhere. (Collocation points are the places where we ask the network to obey the equation. No mesh is needed.)
- Total loss = `lam × physics + boundary`. For now `lam = 1`.
- **No measured data is used anywhere.**
- **The losses are printed live**, every 200 iterations, so you can watch the physics loss and the boundary loss fall. Total loss = physics + boundary (for `lam = 1`).

In [ ]:
def train_forward(its=5000, lr=5e-3, n_col=100, lam=1.0, seed=0, print_every=200, verbose=True):
    """Train a PINN for the cantilever. Prints the losses while it trains. Returns the network and the loss history."""
    need_todos(1, 2, 3)                              # safety net: stops with a clear message if a TODO is not finished
    torch.manual_seed(seed)
    model = make_net()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []                                     # one row every print_every iterations: [iteration, physics loss, boundary loss]
    if verbose:
        print(f"{'Iteration':>10} | {'Total loss':>12} | {'Physics loss':>12} | {'Boundary loss':>14}")
        print("-" * 57)
    for it in range(its):
        x = torch.rand(n_col, 1, requires_grad=True)      # new random collocation points in (0, 1); requires_grad: we differentiate w.r.t. x
        L_phys = physics_loss(model, x)                   # is the beam equation satisfied inside the beam?
        L_bc = bc_loss(model)                             # are the supports respected?
        loss = lam * L_phys + L_bc
        opt.zero_grad()                                   # the four lines of every training step
        loss.backward()
        opt.step()
        if it % print_every == 0 or it == its - 1:
            history.append([it, L_phys.item(), L_bc.item()])
            if verbose:
                print(f"{it:10d} | {loss.item():12.3e} | {L_phys.item():12.3e} | {L_bc.item():14.3e}", flush=True)
    return model, np.array(history)

def evaluate(model, xg=GRID):
    """w, w', w'' (moment) and w''' (shear) of the network on a grid, as numpy arrays"""
    x = torch.tensor(xg.reshape(-1, 1), requires_grad=True)
    w, w1, w2, w3, _ = derivs(model, x)
    return [t.detach().numpy().ravel() for t in (w, w1, w2, w3)]

The figure shows where the collocation points and the boundary conditions sit on the beam.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 2.5))
xcol = np.random.RandomState(1).rand(100)                           # 100 random positions along the beam
ax.plot([0, L], [0, 0], color=NAVY, lw=6, solid_capstyle="butt")
ax.plot(xcol * L, np.zeros(100), "|", color=ORANGE, ms=22, mew=1.4, label="100 collocation points (new each iteration)")
ax.plot([0, L], [0, 0], "s", color=INK, ms=9, label="boundary points")
ax.annotate("fixed end\nw = 0,  w' = 0", (0, 0), (0.0, 0.55), ha="left", fontsize=10.5, color=INK)
ax.annotate("free end\nw'' = 0,  w''' = 0", (L, 0), (L, 0.55), ha="right", fontsize=10.5, color=INK)
ax.set(xlim=(-0.1, L + 0.1), ylim=(-0.5, 1.5), xlabel="x (m)")
ax.set_yticks([])
ax.grid(False)
ax.spines["left"].set_visible(False)
ax.legend(loc="lower center", ncol=2, fontsize=9.5, bbox_to_anchor=(0.5, -0.02))
show(fig, "n1_collocation")

### Train (5,000 iterations, about a minute)
Watch the table fill in while the network trains: both loss terms should fall by many orders of magnitude.

In [ ]:
t0 = time.time()
model, hist = train_forward(its=5000, seed=0)            # prints a row of losses every 200 iterations
print(f"\nTrained in {time.time() - t0:.0f} s")

w, w1, w2, w3 = evaluate(model)                          # the trained network on the 201-point grid
err_w = 100 * rel_l2(w_unif(GRID), w)                    # error of the deflection against the exact solution
print(f"Relative L2 error of the deflection: {err_w:.2f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(hist[:, 0], hist[:, 1] + hist[:, 2], color=TEAL, lw=2, label="total loss")
ax.semilogy(hist[:, 0], hist[:, 1], color=NAVY, lw=2, label="physics loss")
ax.semilogy(hist[:, 0], hist[:, 2], color=ORANGE, lw=2, label="boundary loss")
ax.set(xlabel="iteration", ylabel="loss", title="PINN training: loss against iteration")
ax.legend()
show(fig, "n1_training_loss")

**Read the picture.** The network never saw a deflection, a moment or a shear. It was only asked to satisfy `w''''/8 = 1` and four boundary conditions. The moment `w''` and the shear `−w'''` are further derivatives of the *same* network.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
xm = GRID * L                                                # position in metres
ax[0].plot(xm, to_mm(w_unif(GRID)), color=NAVY, lw=2.4, label="exact")
ax[0].plot(xm, to_mm(w), color=TEAL, lw=2, ls="--", label="PINN")
ax[0].set(xlabel="x (m)", ylabel="mm", title=f"Deflection  (error {err_w:.2f} %)")
ax[0].legend(loc="upper left")
ax[1].plot(xm, to_kNm(M_unif(GRID)), color=NAVY, lw=2.4)
ax[1].plot(xm, to_kNm(w2), color=TEAL, lw=2, ls="--")
ax[1].set(xlabel="x (m)", ylabel="kN·m", title="Moment  w''  (never trained)")
ax[2].plot(xm, to_kN(V_unif(GRID)), color=NAVY, lw=2.4)
ax[2].plot(xm, to_kN(-w3), color=TEAL, lw=2, ls="--")
ax[2].set(xlabel="x (m)", ylabel="kN", title="Shear  −w'''  (never trained)")
show(fig, "n1_forward")

## C · Check it like an engineer
A small error against the exact curve is nice, but an engineer asks: *does the tip deflection match `q L⁴ / 8EI`, and does the fixed-end moment match `q L² / 2`?*

### TODO 4 · Read two numbers off the PINN (two lines)
**What to do.** Fill in `tip_mm` (tip deflection in mm) and `M0_kNm` (fixed-end moment in kN·m) from the PINN.

**Why.** These are the two numbers a designer would use. The moment comes from a *second derivative* of the network, which was never trained on moments.

**How.** `w` and `w2` are numpy arrays holding the network's deflection and `w''` at the 201 grid points. The grid runs from the fixed end to the free end, so the **tip is the last element** (`w[-1]`) and the **fixed end is the first element** (`w2[0]`). Convert scaled values to real units with `to_mm(...)` and `to_kNm(...)`.

<details><summary>Hint (open if stuck)</summary>

`tip_mm = to_mm(w[-1])` and `M0_kNm = to_kNm(w2[0])`
</details>

In [ ]:
tip_mm = None      # TODO 4a: tip deflection of the PINN in mm        (last grid point of w)
M0_kNm = None      # TODO 4b: fixed-end moment of the PINN in kN m    (first grid point of w2)

In [ ]:
exact_tip = Q_SI * L**4 / (8 * EI) * 1e3         # statics: q L^4 / 8EI, in mm
exact_M0 = Q_SI * L**2 / 2 / 1e3                 # statics: q L^2 / 2, in kN m
print(f"tip deflection:    PINN {fmt(tip_mm)} mm      statics {exact_tip:.2f} mm")
print(f"fixed-end moment:  PINN {fmt(M0_kNm)} kN m     statics {exact_M0:.1f} kN m")

In [ ]:
check_tip_and_moment()

---
## Class experiment 1 · Your birth month sets the balance of the loss
*(one training, about 40 seconds, then two numbers into the class form)*

The loss of this PINN has two terms, and we write them with weights that **add up to one**:

`loss = w_physics × physics + w_boundary × boundary`,  with `w_physics + w_boundary = 1`

So far we have used half and half. **Who chose that?** We did, by hand. The physics term is what stops the network doing whatever it likes between the supports — it is the regulariser of this problem — and its weight says how strongly it is applied. Nobody measured it.

Writing the weights so that they add up to one makes the point clear: what matters is how the loss is **shared** between the terms, not how big it is. (With Adam, multiplying every term by the same constant changes very little, because the optimiser rescales by the size of the gradient.)

| Birth month | Jan | Feb | Mar | Apr | May | Jun | Jul | Aug | Sep | Oct | Nov | Dec |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| `w_physics` | 0.0001 | 0.001 | 0.01 | 0.03 | 0.1 | 0.25 | **0.5** | 0.75 | 0.9 | 0.99 | 0.999 | 0.9999 |
| `w_boundary` | 0.9999 | 0.999 | 0.99 | 0.97 | 0.9 | 0.75 | **0.5** | 0.25 | 0.1 | 0.01 | 0.001 | 0.0001 |

Everything else is identical for everybody, including the random start. **Only the balance differs.**

**What to do.** (1) Type your birth month below and run the cell. (2) It prints two numbers. (3) Type those two numbers into the class form (the QR code is on the screen).

**Before you run: predict.** December puts almost everything on the physics and almost nothing on the supports. Will that beam be better or worse than a half-and-half one?

In [ ]:
birth_month = 7        # <- change me: your birth month (1 = January ... 12 = December)

W_PHYS_BY_MONTH = {1: 0.0001, 2: 0.001, 3: 0.01, 4: 0.03, 5: 0.1, 6: 0.25, 7: 0.5, 8: 0.75, 9: 0.9, 10: 0.99, 11: 0.999, 12: 0.9999}
w_phys = W_PHYS_BY_MONTH[birth_month]                # weight on the beam equation
w_bc = 1.0 - w_phys                                  # weight on the supports: the two add up to 1
lam_mine = w_phys / w_bc                             # train_forward takes the ratio, which is the same balance
print(f"month {birth_month}  ->  w_physics = {w_phys:g},  w_boundary = {w_bc:g}   (they add up to {w_phys + w_bc:.4g})\n")

model_c, hist_c = train_forward(its=5000, lam=lam_mine, seed=0, print_every=1000)
w_c, w1_c, w2_c, w3_c = evaluate(model_c)            # the trained network on the grid
err_c = 100 * rel_l2(w_unif(GRID), w_c)              # error of the deflection against the exact beam (%)

print(f"\ntip deflection {to_mm(w_c[-1]):.2f} mm (statics 5.77)     fixed-end moment {to_kNm(w2_c[0]):.1f} kN m (statics 45.0)")
print("\n" + "=" * 62)
print("  Type these two numbers into the class form:")
print(f"     birth month                     {birth_month}")
print(f"     error against the exact beam    {err_c:.1f} %")
print("=" * 62)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6))
ax[0].semilogy(hist_c[:, 0], hist_c[:, 1], color=NAVY, lw=2, label="physics loss (not weighted)")
ax[0].semilogy(hist_c[:, 0], hist_c[:, 2], color=ORANGE, lw=2, label="boundary loss")
ax[0].set(xlabel="iteration", ylabel="loss", title=f"w_physics = {w_phys:g}: which term stops falling?")
ax[0].legend(fontsize=9.5)
ax[1].plot(GRID * L, to_mm(w_unif(GRID)), color=NAVY, lw=2.4, label="exact")
ax[1].plot(GRID * L, to_mm(w_c), color=ORANGE, lw=2, ls="--", label=f"PINN, w_physics = {w_phys:g}")
ax[1].set(xlabel="x (m)", ylabel="deflection (mm)", title=f"Deflection: error {err_c:.1f} %")
ax[1].legend(fontsize=9.5)
show(fig)

**While the class plot comes up**, two things to look at in your own run:

1. In the left-hand panel, which of the two loss terms stops falling? At the extremes of the table one of them is stuck from the first few hundred iterations.
2. In the right-hand panel, what does the beam do wrong — does it miss the supports, or the curvature?

**Then, on the class plot:** is there one best weight, or a range that all work? And how wide is that range?

---
### After the course
The solution notebook has three further experiments that there is no time for today, and which take a few minutes each to run:

* **D1** — the same problem in SI units instead of scaled units, where the optimiser sees only the physics term.
* **D2** — the weight sweep of today's class experiment, run by one machine over four weights.
* **D3** — three noisy readings with and without the physics term, repeated over five random starts.